In [29]:
from pyspark.sql import SparkSession, functions as f

In [30]:
from pathlib import Path
root =  Path.cwd() / 'work'
data_path = root / 'data' / 'raw' / 'infra_ed_basica_2024.csv'
print(data_path)

/home/jovyan/work/data/raw/infra_ed_basica_2024.csv


In [31]:
spark = SparkSession.builder.appName("Analise_INEP").getOrCreate()

In [51]:
df_censo_escolar = spark.read.csv(str(data_path), header=True, sep=';')
df_censo_escolar.show(5)

+------------+---------+---------+--------+-----+-----+--------------------+------------+---------------------+---------------------+-------------------+-------------------+-----------------+--------------+---------------+---------------+--------------------+-----------+--------------------+-----------+--------------+---------------------------+--------------+---------------------------+--------------------+-----------+---------------+-----------+--------+------+-----------+-------------------------+-----------------+--------------------+---------------------+------------------------------+----------------------------+---------------------------+----------------------+-------------------------+-------------------------+----------------------------+---------------------------+-------------------------+----------------------------+--------------------------+---------------------------+----------------------------+------------------------------+----------------------------+---------------

In [52]:
df_censo_escolar = df_censo_escolar.filter((f.col('TP_SITUACAO_FUNCIONAMENTO') == '1'))

In [53]:
prefixos_infra = ('IN_AGUA', 'IN_ENERGIA', 'IN_ESGOTO', 'IN_LIXO', 'IN_INTERNET', 'IN_BANHEIRO', 'IN_ACESSABILIDADE', 'IN_COZINHA', 'IN_REFEITORIO', 'IN_BIBLIOTECA', 'IN_QUADRA_ESPORTES', 'IN_ALIMENTACAO', 'IN_LABORATORIO')
colunas = df_censo_escolar.columns

colunas_infra = []
for coluna in colunas:
    if coluna.startswith(prefixos_infra):
        colunas_infra.append(coluna)

colunas_final = ['NO_REGIAO','CO_MUNICIPIO', 'NO_MUNICIPIO', 'SG_UF'] + colunas_infra
df_censo_escolar = df_censo_escolar.select(colunas_final)
df_censo_escolar.show(5)

+---------+------------+--------------------+-----+---------------+--------------------+----------------------+---------------+-----------------+-------------------+------------------+-----------------------+-------------------------+--------------------+----------------------+----------------------+-----------------------+---------------------+---------------+---------------------+----------------------+--------------+---------------+-----------------------------+---------------------------+-----------+--------------+---------------+------------------------+--------------------+-------------+--------------------------+----------+-----------------------+--------------------------+------------------------+------------------+--------------------------+-----------------------------+-------------+-----------+------------------+--------------------------+------------------------+----------------------+--------------+
|NO_REGIAO|CO_MUNICIPIO|        NO_MUNICIPIO|SG_UF|IN_AGUA_POTAVEL|IN_AGUA

In [54]:
df_censo_escolar.dropDuplicates(['NO_REGIAO']).show()

+------------+------------+--------------------+-----+---------------+--------------------+----------------------+---------------+-----------------+-------------------+------------------+-----------------------+-------------------------+--------------------+----------------------+----------------------+-----------------------+---------------------+---------------+---------------------+----------------------+--------------+---------------+-----------------------------+---------------------------+-----------+--------------+---------------+------------------------+--------------------+-------------+--------------------------+----------+-----------------------+--------------------------+------------------------+------------------+--------------------------+-----------------------------+-------------+-----------+------------------+--------------------------+------------------------+----------------------+--------------+
|   NO_REGIAO|CO_MUNICIPIO|        NO_MUNICIPIO|SG_UF|IN_AGUA_POTAVEL|I

In [55]:
df_censo_escolar = df_censo_escolar.filter((f.col('NO_REGIAO').isin('Nordeste', 'Sudeste')))
df_censo_escolar.show(5)

+---------+------------+------------+-----+---------------+--------------------+----------------------+---------------+-----------------+-------------------+------------------+-----------------------+-------------------------+--------------------+----------------------+----------------------+-----------------------+---------------------+---------------+---------------------+----------------------+--------------+---------------+-----------------------------+---------------------------+-----------+--------------+---------------+------------------------+--------------------+-------------+--------------------------+----------+-----------------------+--------------------------+------------------------+------------------+--------------------------+-----------------------------+-------------+-----------+------------------+--------------------------+------------------------+----------------------+--------------+
|NO_REGIAO|CO_MUNICIPIO|NO_MUNICIPIO|SG_UF|IN_AGUA_POTAVEL|IN_AGUA_REDE_PUBLICA|IN

In [56]:
for col in df_censo_escolar.columns:
    df_censo_escolar = df_censo_escolar.withColumnRenamed(col, col.lower())

In [57]:
df_censo_escolar.printSchema()

root
 |-- no_regiao: string (nullable = true)
 |-- co_municipio: string (nullable = true)
 |-- no_municipio: string (nullable = true)
 |-- sg_uf: string (nullable = true)
 |-- in_agua_potavel: string (nullable = true)
 |-- in_agua_rede_publica: string (nullable = true)
 |-- in_agua_poco_artesiano: string (nullable = true)
 |-- in_agua_cacimba: string (nullable = true)
 |-- in_agua_fonte_rio: string (nullable = true)
 |-- in_agua_inexistente: string (nullable = true)
 |-- in_agua_carro_pipa: string (nullable = true)
 |-- in_energia_rede_publica: string (nullable = true)
 |-- in_energia_gerador_fossil: string (nullable = true)
 |-- in_energia_renovavel: string (nullable = true)
 |-- in_energia_inexistente: string (nullable = true)
 |-- in_esgoto_rede_publica: string (nullable = true)
 |-- in_esgoto_fossa_septica: string (nullable = true)
 |-- in_esgoto_fossa_comum: string (nullable = true)
 |-- in_esgoto_fossa: string (nullable = true)
 |-- in_esgoto_inexistente: string (nullable = true)

In [58]:
colunas = [f.col(c).cast('integer').alias(c) if c.startswith('in_') else f.col(c)for c in df_censo_escolar.columns]
df_censo_escolar = df_censo_escolar.select(colunas)

In [59]:
df_censo_escolar.printSchema()

root
 |-- no_regiao: string (nullable = true)
 |-- co_municipio: string (nullable = true)
 |-- no_municipio: string (nullable = true)
 |-- sg_uf: string (nullable = true)
 |-- in_agua_potavel: integer (nullable = true)
 |-- in_agua_rede_publica: integer (nullable = true)
 |-- in_agua_poco_artesiano: integer (nullable = true)
 |-- in_agua_cacimba: integer (nullable = true)
 |-- in_agua_fonte_rio: integer (nullable = true)
 |-- in_agua_inexistente: integer (nullable = true)
 |-- in_agua_carro_pipa: integer (nullable = true)
 |-- in_energia_rede_publica: integer (nullable = true)
 |-- in_energia_gerador_fossil: integer (nullable = true)
 |-- in_energia_renovavel: integer (nullable = true)
 |-- in_energia_inexistente: integer (nullable = true)
 |-- in_esgoto_rede_publica: integer (nullable = true)
 |-- in_esgoto_fossa_septica: integer (nullable = true)
 |-- in_esgoto_fossa_comum: integer (nullable = true)
 |-- in_esgoto_fossa: integer (nullable = true)
 |-- in_esgoto_inexistente: integer (

In [60]:
df_censo_escolar.select([f.count(f.when(f.isnan(c) | f.col(c).isNull(), c)).alias(c) for c in df_censo_escolar.columns]).show()

+---------+------------+------------+-----+---------------+--------------------+----------------------+---------------+-----------------+-------------------+------------------+-----------------------+-------------------------+--------------------+----------------------+----------------------+-----------------------+---------------------+---------------+---------------------+----------------------+--------------+---------------+-----------------------------+---------------------------+-----------+--------------+---------------+------------------------+--------------------+-------------+--------------------------+----------+-----------------------+--------------------------+------------------------+------------------+--------------------------+-----------------------------+-------------+-----------+------------------+--------------------------+------------------------+----------------------+--------------+
|no_regiao|co_municipio|no_municipio|sg_uf|in_agua_potavel|in_agua_rede_publica|in

In [61]:
expressoes_agregacao = []

for coluna in colunas_infra:
    expressoes_agregacao.append(f.avg(coluna).alias(f"taxa_{coluna.lower()}"))
    expressoes_agregacao.append(f.stddev(coluna).alias(f"desvio_{coluna.lower()}"))

df_censo_escolar.groupBy("co_municipio").agg(*expressoes_agregacao).show()

+------------+--------------------+----------------------+-------------------------+---------------------------+---------------------------+-----------------------------+--------------------+----------------------+----------------------+------------------------+------------------------+--------------------------+-----------------------+-------------------------+----------------------------+------------------------------+------------------------------+--------------------------------+-------------------------+---------------------------+---------------------------+-----------------------------+---------------------------+-----------------------------+----------------------------+------------------------------+--------------------------+----------------------------+--------------------+----------------------+--------------------------+----------------------------+---------------------------+-----------------------------+--------------------+---------------------+--------------------+----